In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop         = fm.FontProperties(fname=font_path,      size=11)
font_prop_small   = fm.FontProperties(fname=font_path,      size=7)
font_prop_numbers = fm.FontProperties(fname=font_path,      size=9)
font_prop_title   = fm.FontProperties(fname=font_path_bold, size=14)

FIG_WIDTH = 16
CELL_BORDER_COLOR = 'white'

folders_filtered = ["Chronos-2", "Moirai-2", "Lag-Llama", "TimeGPT", "TimesFM", "TabPFN-TS"]

# ---------------------------------------------------------
# Location code -> abbreviation mapping (FluSight locations.csv, shared across challenges)
# ---------------------------------------------------------
locations_ref = pd.read_csv('../auxiliary-files/locations.csv', dtype={'location': str})
loc_to_abbr = dict(zip(locations_ref['location'], locations_ref['abbreviation']))

## 1. Function to collect negative values + correctly-normalized pivot tables for a given challenge\n\n- `pivot_date`: normalized by `n_instances` (total rows in a single reference-date file, across all locations/horizons/quantiles).\n- `pivot_loc`: normalized by the total instance count **specific to each (model, location)**, summed across all reference dates — since a single file's `n_instances` reflects the total across *all* locations for *one* date, not the total across *all* dates for *one* location.

In [ ]:
def build_negative_values_pivots(challenge):
    """
    Legge tutti i file model-output per un dato challenge (FluSight o CovidHub),
    raccoglie i valori negativi e costruisce pivot_date/pivot_loc correttamente normalizzati.
    """
    negative_rows = []
    all_rows_count = []  # per calcolare il totale istanze per (model, location), su tutte le reference_date

    for folder in folders_filtered:
        files = [f for f in Path(f"../model-output/{challenge}/{folder}").resolve().iterdir()
                 if f.is_file() and f.suffix == '.csv'
                 and '2025-11-22' <= f.name[:10] <= '2026-05-23']  # evaluation window


        for file in files:
            df = pd.read_csv(file)

            # conta TUTTE le righe (non solo i negativi) per location, per il denominatore corretto
            loc_counts = df.groupby('location').size().reset_index(name='n_total_loc')
            loc_counts['model'] = folder
            all_rows_count.append(loc_counts)

            # filtra solo i negativi
            neg_mask = df['value'] < 0
            # TimeGPT 2026-04-25 (retrospective round): the generation workflow already set negative
            # quantiles to 0. TimeGPT never outputs exact zeros otherwise (<= 1 per file in all other
            # rounds), so exact zeros in this file are the clipped negatives.
            if folder == 'TimeGPT' and file.name.startswith('2026-04-25'):
                neg_mask = df['value'] <= 0
            neg_df = df[neg_mask].copy()
            if not neg_df.empty:
                neg_df['model'] = folder
                neg_df['n_instances'] = df.shape[0]   # totale istanze per quella reference_date
                negative_rows.append(neg_df)

    negative_values_df = pd.concat(negative_rows, ignore_index=True)
    negative_values_df = negative_values_df[
        ['model', 'reference_date', 'target_end_date', "location", 'horizon', 'output_type_id', 'value', 'n_instances']
    ]

    # totale istanze per (model, location), sommando su tutte le reference_date
    loc_totals_df = pd.concat(all_rows_count, ignore_index=True)
    n_instances_by_loc = (
        loc_totals_df
        .groupby(['model', 'location'])['n_total_loc']
        .sum()
        .reset_index()
    )

    # mappa location -> abbreviation
    negative_values_df['location'] = negative_values_df['location'].astype(str).str.zfill(2)
    negative_values_df['location_abbr'] = negative_values_df['location'].map(loc_to_abbr)

    n_instances_by_loc['location'] = n_instances_by_loc['location'].astype(str).str.zfill(2)
    n_instances_by_loc['location_abbr'] = n_instances_by_loc['location'].map(loc_to_abbr)

    # sanity check: righe con chiave duplicata
    key_cols = ['model', 'reference_date', 'target_end_date', 'horizon', 'output_type_id', 'location']
    dupes = negative_values_df[negative_values_df.duplicated(subset=key_cols, keep=False)]
    if len(dupes) > 0:
        print(f"[{challenge}] ATTENZIONE: {len(dupes)} righe con chiave duplicata trovate")

    # ---------------------------------------------------------
    # pivot_date: normalizza per n_instances (totale per reference_date)
    # ---------------------------------------------------------
    norm_factor = negative_values_df['n_instances'].unique()[0]

    pivot_date = negative_values_df.pivot_table(
        index='model', columns='reference_date', values='value',
        aggfunc='size', fill_value=0
    ).reindex(folders_filtered) / norm_factor * 100

    # ---------------------------------------------------------
    # pivot_loc: normalizza per il totale istanze SPECIFICO di ciascuna location e modello
    # ---------------------------------------------------------
    pivot_loc_counts = negative_values_df.pivot_table(
        index='model', columns='location_abbr', values='value',
        aggfunc='size', fill_value=0
    )

    pivot_loc_totals = n_instances_by_loc.pivot_table(
        index='model', columns='location_abbr', values='n_total_loc', fill_value=0
    )

    pivot_loc_counts, pivot_loc_totals = pivot_loc_counts.align(pivot_loc_totals, join='outer', fill_value=0)

    pivot_loc = (pivot_loc_counts / pivot_loc_totals.replace(0, np.nan) * 100).reindex(folders_filtered)

    # reindex su TUTTE le 53 location (anche quelle senza alcun valore negativo,
    # che altrimenti sparirebbero come colonna invece di comparire come 0%/NaN)
    full_locations = locations_ref['abbreviation'].tolist()
    pivot_loc = pivot_loc.reindex(columns=full_locations)

    pivot_date = pivot_date.replace(0, np.nan)
    pivot_loc = pivot_loc.replace(0, np.nan)

    return pivot_date, pivot_loc, negative_values_df, n_instances_by_loc

## 2. Build pivots for both challenges

In [ ]:
pivot_date_flu, pivot_loc_flu, negative_values_df_flu, n_instances_by_loc_flu = build_negative_values_pivots('FluSight')
pivot_date_covid, pivot_loc_covid, negative_values_df_covid, n_instances_by_loc_covid = build_negative_values_pivots('CovidHub')

pivot_date_flu

In [ ]:
pivot_loc_covid

## 3. Plot combined figure: 4 panels (A/B = FluSight, C/D = CovidHub)

In [ ]:
def style_heatmap_axis(ax, cbar_label):
    """Applica font_prop a tutti gli elementi testuali di una heatmap"""
    plt.setp(ax.get_xticklabels(), fontproperties=font_prop_small)
    plt.setp(ax.get_yticklabels(), fontproperties=font_prop)
    ax.set_xlabel(ax.get_xlabel(), fontproperties=font_prop)
    ax.set_ylabel(ax.get_ylabel(), fontproperties=font_prop)
    ax.set_title(ax.get_title(), fontproperties=font_prop_title, loc='left')

    cbar = ax.collections[0].colorbar
    cbar.set_label(cbar_label, fontproperties=font_prop)
    plt.setp(cbar.ax.get_yticklabels(), fontproperties=font_prop)


# colormap personalizzata: bianco panna (0/NaN) -> rosso (max)
CREAM = '#F5EFE6'
reds_colors = plt.cm.Reds(np.linspace(0, 1, 256))
reds_colors[0] = mcolors.to_rgba(CREAM)
cmap_with_nan = mcolors.LinearSegmentedColormap.from_list('reds_cream', reds_colors)
cmap_with_nan.set_bad(color=CREAM)

fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 10), dpi=300)

panels = [
    (axes[0, 0], pivot_date_flu,   'A. FluSight — percentage of negative values by reference date', 45),
    (axes[0, 1], pivot_loc_flu,    'B. FluSight — percentage of negative values by location',        90),
    (axes[1, 0], pivot_date_covid, 'C. CovidHub — percentage of negative values by reference date',  45),
    (axes[1, 1], pivot_loc_covid,  'D. CovidHub — percentage of negative values by location',         90),
]

for ax, pivot, title, rotation in panels:
    sns.heatmap(
        pivot, cmap=cmap_with_nan, linewidths=0.5, linecolor=CELL_BORDER_COLOR,
        cbar_kws={'label': 'Percentage of negative values (%)'}, ax=ax,
        xticklabels=True, yticklabels=True   # <-- aggiunto qui
    )
    ax.set_xlabel(' ')
    ax.set_ylabel(' ')
    style_heatmap_axis(ax, 'Percentage of negative values (%)')
    ax.tick_params(axis='x', rotation=rotation)
    ax.set_title(title, fontproperties=font_prop_title, loc='left', pad=10)

plt.tight_layout()
plt.savefig('../model-evaluation/figures/negative_values_summary_combined.png', dpi=300, bbox_inches='tight')
plt.savefig('../model-evaluation/figures/negative_values_summary_combined.svg', dpi=300, bbox_inches='tight')
plt.show()